# Creative Advertising Multi-Agent System — v2
## AI Agentic Bootcamp — Part B (Upgraded)

This notebook applies **CrewAI best practices** to the OpenAI Agents SDK:
- **Structured outputs** via Pydantic models (`output_type`)
- **Role / Goal / Backstory** agent instructions
- **4-agent pipeline** with a new Media Planner agent
- **Proper `handoff` wiring** between agents
- **Retry logic** with exponential backoff
- **Input guardrails** (length + content validation)
- **Token & cost tracking** per agent
- **Structured JSON output** file

### Agent Pipeline
```
User Prompt
    ↓
Creative Director  →  CreativeBrief (Pydantic)
    ↓
Strategist         →  CampaignStrategy (Pydantic)
    ↓
Copywriter         →  AdCopy (Pydantic)
    ↓
Media Planner      →  MediaPlan (Pydantic)
```

### Test Prompt
> *"Launch a campaign for a new eco-friendly water bottle in Bali"*

## 1. Setup & Imports

In [1]:
# Install dependencies (run once)
# !pip install openai-agents pydantic python-dotenv

import os
import json
import asyncio
from agents import Agent, Runner, handoff
from pydantic import BaseModel, ConfigDict
from ad_models import CreativeBrief, CampaignStrategy, AdCopy, ChannelAllocation, MediaPlan
from dotenv import load_dotenv
load_dotenv()

print("All imports successful.")

All imports successful.


## 2. Configure API Key

In [2]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Please set it before running the pipeline.")
else:
    print(f"API key loaded: {api_key[:8]}...")

API key loaded: sk-proj-...


## 3. Structured Output Models (Pydantic)

Each agent returns a **typed Pydantic model** instead of raw text.
This mirrors CrewAI's `output_pydantic` pattern and makes downstream
parsing reliable and IDE-friendly.

In [3]:
print("Models imported from ad_models.py")

Models imported from ad_models.py


## 4. Define the Four Agents

Each agent uses the **Role / Goal / Backstory** instruction pattern from CrewAI:
- **Role** — what the agent *is*
- **Goal** — what it must *achieve*
- **Backstory** — context that shapes its *judgment*
- **Output format** — explicit JSON schema tied to the Pydantic model

Agents are wired with `handoffs` so the SDK knows the intended flow.

In [4]:
# ─────────────────────────────────────────────────────────────
# Agent 4: Media Planner  (defined first — referenced by copywriter handoff)
# ─────────────────────────────────────────────────────────────
media_planner = Agent(
    name="Media Planner",
    model="gpt-4o-mini",
    output_type=MediaPlan,
    instructions=(
        "ROLE: You are a Senior Media Planner at a full-service advertising agency.\n"
        "GOAL: Translate a finished creative campaign into a concrete, budget-allocated "
        "media plan with measurable KPIs and a realistic launch timeline.\n"
        "BACKSTORY: You have placed campaigns across Instagram, TikTok, YouTube, OOH, "
        "and influencer networks. You think in CPM, ROAS, and reach curves. You balance "
        "awareness spend against conversion spend based on the brand's maturity stage.\n\n"
        "OUTPUT INSTRUCTIONS: Return a valid JSON object matching this schema:\n"
        "{\n"
        "  \"total_budget_usd\": <integer, recommended total budget>,\n"
        "  \"channel_allocations\": [{\"channel\": \"Instagram\", \"percent_of_budget\": 40}, ...],\n"
        "  \"campaign_timeline_weeks\": <integer>,\n"
        "  \"kpis\": [\"KPI 1\", ...],\n"
        "  \"launch_milestones\": [\"Week N: action\", ...]\n"
        "}\n"
        "Return ONLY the JSON object. No markdown fences, no extra text."
    ),
)

print("Media Planner agent defined.")

Media Planner agent defined.


In [5]:
# ─────────────────────────────────────────────────────────────
# Agent 3: Copywriter
# ─────────────────────────────────────────────────────────────
copywriter = Agent(
    name="Copywriter",
    model="gpt-4o-mini",
    output_type=AdCopy,
    handoffs=[handoff(media_planner)],
    instructions=(
        "ROLE: You are an award-winning Copywriter known for culturally resonant, "
        "conversion-driven ad copy.\n"
        "GOAL: Produce a complete suite of ad copy assets — taglines, social captions, "
        "video scripts, email copy, and print — that are tightly aligned with the "
        "creative brief and strategy you receive.\n"
        "BACKSTORY: You have written campaigns for global consumer brands. Your copy "
        "has won Cannes Lions. You never use clichés. You match tone to platform "
        "(punchy on Instagram, narrative on email, sensory for print).\n\n"
        "OUTPUT INSTRUCTIONS: Return a valid JSON object matching this schema:\n"
        "{\n"
        "  \"hero_tagline\": \"<short, punchy, memorable>\",\n"
        "  \"instagram_caption\": \"<150 chars max + 5 hashtags>\",\n"
        "  \"video_script\": \"<15-second hook + CTA>\",\n"
        "  \"email_subject\": \"<subject line>\",\n"
        "  \"email_preview\": \"<preview text, 90 chars max>\",\n"
        "  \"print_headline\": \"<print ad headline>\",\n"
        "  \"print_body\": \"<50-word supporting body copy>\"\n"
        "}\n"
        "Return ONLY the JSON object. No markdown fences, no extra text."
    ),
)

print("Copywriter agent defined.")

Copywriter agent defined.


In [6]:
# ─────────────────────────────────────────────────────────────
# Agent 2: Strategist
# ─────────────────────────────────────────────────────────────
strategist = Agent(
    name="Strategist",
    model="gpt-4o-mini",
    output_type=CampaignStrategy,
    handoffs=[handoff(copywriter)],
    instructions=(
        "ROLE: You are a Brand Strategist specialising in digital and experiential marketing.\n"
        "GOAL: Transform a creative brief into an actionable strategic plan — "
        "audience profile, messaging pillars, channel mix, and campaign hooks — "
        "that the Copywriter can execute immediately.\n"
        "BACKSTORY: You have led go-to-market strategies for lifestyle, wellness, "
        "and sustainability brands across South-East Asia. You combine cultural "
        "intelligence with data-driven channel selection. You know that a strategy "
        "is only as good as its specificity.\n\n"
        "OUTPUT INSTRUCTIONS: Return a valid JSON object matching this schema:\n"
        "{\n"
        "  \"target_audience\": \"<one paragraph profile>\",\n"
        "  \"messaging_pillars\": [\"Pillar 1\", \"Pillar 2\", \"Pillar 3\"],\n"
        "  \"recommended_channels\": [\"channel 1\", ...],\n"
        "  \"campaign_hooks\": [\"hook 1\", ...]\n"
        "}\n"
        "Return ONLY the JSON object. No markdown fences, no extra text."
    ),
)

print("Strategist agent defined.")

Strategist agent defined.


In [7]:
# ─────────────────────────────────────────────────────────────
# Agent 1: Creative Director
# ─────────────────────────────────────────────────────────────
creative_director = Agent(
    name="Creative Director",
    model="gpt-4o-mini",
    output_type=CreativeBrief,
    handoffs=[handoff(strategist)],
    instructions=(
        "ROLE: You are a Senior Creative Director at a top-tier global advertising agency.\n"
        "GOAL: Receive a raw campaign request and produce a sharp, inspiring creative "
        "brief that sets the vision for the entire campaign team.\n"
        "BACKSTORY: You have led campaigns for Nike, Patagonia, and emerging DTC brands. "
        "You know that the best briefs are opinionated — they make a clear creative bet "
        "rather than trying to please everyone. You always consider local culture, "
        "emotional resonance, and brand differentiation before you write a single word.\n\n"
        "OUTPUT INSTRUCTIONS: Return a valid JSON object matching this schema:\n"
        "{\n"
        "  \"campaign_name\": \"<evocative campaign title>\",\n"
        "  \"tagline_concept\": \"<core tagline idea>\",\n"
        "  \"core_brand_message\": \"<1-2 sentence brand truth>\",\n"
        "  \"visual_direction\": \"<mood, palette, aesthetic, style>\",\n"
        "  \"objectives\": [\"objective 1\", \"objective 2\", ...]\n"
        "}\n"
        "Return ONLY the JSON object. No markdown fences, no extra text."
    ),
)

print("Creative Director agent defined.")
print("\nHandoff chain: Creative Director → Strategist → Copywriter → Media Planner")

Creative Director agent defined.

Handoff chain: Creative Director → Strategist → Copywriter → Media Planner


## 5. Input Guardrails

Validate the campaign prompt *before* spending any API calls.

In [8]:
BANNED_KEYWORDS = {"spam", "mislead", "fake", "scam", "illegal"}

def validate_campaign_prompt(prompt: str) -> str:
    """Validate and sanitise the campaign prompt before pipeline execution."""
    prompt = prompt.strip()

    if len(prompt) < 10:
        raise ValueError(
            f"Campaign prompt too short ({len(prompt)} chars). "
            "Provide at least 10 characters."
        )
    if len(prompt) > 500:
        raise ValueError(
            f"Campaign prompt too long ({len(prompt)} chars). "
            "Keep it under 500 characters."
        )

    lower = prompt.lower()
    hits = [kw for kw in BANNED_KEYWORDS if kw in lower]
    if hits:
        raise ValueError(
            f"Campaign prompt contains disallowed content: {hits}"
        )

    return prompt


# --- Quick guardrail tests ---
try:
    validate_campaign_prompt("")
except ValueError as e:
    print(f"[GUARDRAIL] Caught expected error: {e}")

try:
    validate_campaign_prompt("x" * 501)
except ValueError as e:
    print(f"[GUARDRAIL] Caught expected error: {e}")

try:
    validate_campaign_prompt("Launch a scam product campaign")
except ValueError as e:
    print(f"[GUARDRAIL] Caught expected error: {e}")

valid = validate_campaign_prompt("Launch a campaign for a new eco-friendly water bottle in Bali")
print(f"[GUARDRAIL] Valid prompt accepted: '{valid[:50]}...'")

[GUARDRAIL] Caught expected error: Campaign prompt too short (0 chars). Provide at least 10 characters.
[GUARDRAIL] Caught expected error: Campaign prompt too long (501 chars). Keep it under 500 characters.
[GUARDRAIL] Caught expected error: Campaign prompt contains disallowed content: ['scam']
[GUARDRAIL] Valid prompt accepted: 'Launch a campaign for a new eco-friendly water bot...'


## 6. Retry Helper with Exponential Backoff

In [9]:
async def run_agent_with_retry(agent: Agent, input_text: str, max_retries: int = 3):
    """
    Run an agent with exponential-backoff retry on transient errors.

    Args:
        agent: The Agent instance to run.
        input_text: The prompt/context string.
        max_retries: Maximum number of attempts before re-raising.

    Returns:
        The RunResult from Runner.run().
    """
    last_exc = None
    for attempt in range(max_retries):
        try:
            result = await Runner.run(agent, input=input_text)
            return result
        except Exception as exc:
            last_exc = exc
            if attempt < max_retries - 1:
                wait = 2 ** attempt          # 1s, 2s, 4s …
                print(
                    f"  [RETRY] {agent.name} attempt {attempt + 1}/{max_retries} "
                    f"failed ({type(exc).__name__}: {exc}). "
                    f"Retrying in {wait}s…"
                )
                await asyncio.sleep(wait)
            else:
                print(f"  [RETRY] {agent.name} exhausted {max_retries} attempts.")

    raise last_exc


print("Retry helper defined.")

Retry helper defined.


## 7. Token & Cost Tracking Helper

In [10]:
# gpt-4o-mini pricing (as of 2025) — update if rates change
COST_PER_1K_INPUT  = 0.000150   # USD per 1,000 input tokens
COST_PER_1K_OUTPUT = 0.000600   # USD per 1,000 output tokens

def extract_token_usage(result) -> dict:
    """
    Extract token usage from a Runner result.
    Returns a dict with input_tokens, output_tokens, and estimated_cost_usd.
    """
    usage = {"input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0}

    try:
        # OpenAI Agents SDK stores usage in result.raw_responses
        for response in getattr(result, "raw_responses", []):
            u = getattr(response, "usage", None)
            if u:
                usage["input_tokens"]  += getattr(u, "input_tokens",  0)
                usage["output_tokens"] += getattr(u, "output_tokens", 0)
    except Exception:
        pass   # usage tracking is best-effort

    usage["estimated_cost_usd"] = (
        usage["input_tokens"]  / 1000 * COST_PER_1K_INPUT +
        usage["output_tokens"] / 1000 * COST_PER_1K_OUTPUT
    )
    return usage


def print_token_report(token_log: dict):
    """Pretty-print a token usage report across all agents."""
    print("\n" + "─" * 50)
    print("TOKEN USAGE REPORT")
    print("─" * 50)
    total_in, total_out, total_cost = 0, 0, 0.0
    for agent_name, u in token_log.items():
        print(
            f"  {agent_name:<20}  "
            f"in={u['input_tokens']:>5}  "
            f"out={u['output_tokens']:>5}  "
            f"~${u['estimated_cost_usd']:.5f}"
        )
        total_in   += u["input_tokens"]
        total_out  += u["output_tokens"]
        total_cost += u["estimated_cost_usd"]
    print("─" * 50)
    print(
        f"  {'TOTAL':<20}  "
        f"in={total_in:>5}  "
        f"out={total_out:>5}  "
        f"~${total_cost:.5f}"
    )


print("Token tracking helpers defined.")

Token tracking helpers defined.


## 8. Build the 4-Agent Sequential Pipeline

Each agent:
1. Receives the **accumulated context** of all prior agents
2. Returns a **typed Pydantic object** (not raw text)
3. Is wired via **`handoff`** declarations on the agent definition
4. Is called through the **retry wrapper**
5. Logs **token usage**

In [11]:
async def run_advertising_pipeline_v2(campaign_prompt: str) -> dict:
    """
    Run the upgraded 4-agent advertising pipeline with structured outputs,
    retry logic, guardrails, and token tracking.

    Args:
        campaign_prompt: The initial campaign request from the user.

    Returns:
        dict with keys: creative_brief, strategy, copy, media_plan, token_usage
    """
    # ── Guardrail ──────────────────────────────────────────────
    campaign_prompt = validate_campaign_prompt(campaign_prompt)

    results    = {}
    token_log  = {}

    print("=" * 62)
    print("  CREATIVE ADVERTISING PIPELINE v2  (4 agents)")
    print("=" * 62)
    print(f"  Campaign: {campaign_prompt}\n")

    # ── Step 1: Creative Director ──────────────────────────────
    print("[1/4] Creative Director — drafting creative brief…")
    result1 = await run_agent_with_retry(
        creative_director,
        input_text=f"Campaign request: {campaign_prompt}"
    )
    brief: CreativeBrief = result1.final_output
    results["creative_brief"] = brief.model_dump()
    token_log["Creative Director"] = extract_token_usage(result1)

    print(f"\n  Campaign name : {brief.campaign_name}")
    print(f"  Tagline       : {brief.tagline_concept}")
    print(f"  Message       : {brief.core_brand_message}")
    print(f"  Objectives    : {brief.objectives}")

    # ── Step 2: Strategist ─────────────────────────────────────
    print("\n[2/4] Strategist — developing campaign strategy…")
    strategy_input = (
        f"Original campaign request: {campaign_prompt}\n\n"
        f"Creative Brief (JSON):\n{json.dumps(brief.model_dump(), indent=2)}"
    )
    result2 = await run_agent_with_retry(strategist, input_text=strategy_input)
    strategy: CampaignStrategy = result2.final_output
    results["strategy"] = strategy.model_dump()
    token_log["Strategist"] = extract_token_usage(result2)

    print(f"\n  Audience      : {strategy.target_audience[:80]}…")
    print(f"  Pillars       : {strategy.messaging_pillars}")
    print(f"  Channels      : {strategy.recommended_channels}")

    # ── Step 3: Copywriter ─────────────────────────────────────
    print("\n[3/4] Copywriter — producing ad copy suite…")
    copy_input = (
        f"Original campaign request: {campaign_prompt}\n\n"
        f"Creative Brief (JSON):\n{json.dumps(brief.model_dump(), indent=2)}\n\n"
        f"Campaign Strategy (JSON):\n{json.dumps(strategy.model_dump(), indent=2)}"
    )
    result3 = await run_agent_with_retry(copywriter, input_text=copy_input)
    copy: AdCopy = result3.final_output
    results["copy"] = copy.model_dump()
    token_log["Copywriter"] = extract_token_usage(result3)

    print(f"\n  Hero tagline  : {copy.hero_tagline}")
    print(f"  Email subject : {copy.email_subject}")
    print(f"  Print headline: {copy.print_headline}")

    # ── Step 4: Media Planner ──────────────────────────────────
    print("\n[4/4] Media Planner — building media plan & KPIs…")
    media_input = (
        f"Original campaign request: {campaign_prompt}\n\n"
        f"Creative Brief (JSON):\n{json.dumps(brief.model_dump(), indent=2)}\n\n"
        f"Campaign Strategy (JSON):\n{json.dumps(strategy.model_dump(), indent=2)}\n\n"
        f"Ad Copy (JSON):\n{json.dumps(copy.model_dump(), indent=2)}"
    )
    result4 = await run_agent_with_retry(media_planner, input_text=media_input)
    plan: MediaPlan = result4.final_output
    results["media_plan"] = plan.model_dump()
    token_log["Media Planner"] = extract_token_usage(result4)

    print(f"\n  Total budget  : ${plan.total_budget_usd:,}")
    print(f"  Timeline      : {plan.campaign_timeline_weeks} weeks")
    print(f"  KPIs          : {plan.kpis}")
    alloc_summary = {a.channel: f"{a.percent_of_budget}%" for a in plan.channel_allocations}
    print(f"  Allocations   : {alloc_summary}")

    # ── Summary ────────────────────────────────────────────────
    results["token_usage"] = token_log
    print_token_report(token_log)

    print("\n" + "=" * 62)
    print("  PIPELINE COMPLETE — all 4 agents produced structured output")
    print("=" * 62)

    return results


print("Pipeline v2 function defined.")

Pipeline v2 function defined.


## 9. Run the Pipeline — Test Prompt

> **Test Prompt**: *"Launch a campaign for a new eco-friendly water bottle in Bali"*

In [12]:
TEST_PROMPT = "Launch a campaign for a new eco-friendly water bottle in Bali"

results = await run_advertising_pipeline_v2(TEST_PROMPT)

  CREATIVE ADVERTISING PIPELINE v2  (4 agents)
  Campaign: Launch a campaign for a new eco-friendly water bottle in Bali

[1/4] Creative Director — drafting creative brief…

  Campaign name : Bali’s Pure Flow
  Tagline       : Refresh Sustainably, Live Consciously
  Message       : Embrace Bali's natural beauty with every sip from your eco-friendly water bottle, protecting the planet while enjoying pure hydration.
  Objectives    : ["Raise awareness of the eco-friendly water bottle among Bali's tourists and locals", 'Position the brand as a leader in sustainable living', 'Drive sales through engaging social media campaigns and local partnerships']

[2/4] Strategist — developing campaign strategy…

  Audience      : The target audience includes environmentally conscious tourists aged 25-45 visit…
  Pillars       : ["Sustainability: Highlight the product's eco-friendly materials and impact on reducing plastic waste.", "Lifestyle: Position the water bottle as an essential accessory for an

## 10. Inspect Structured Outputs

Each result is now a typed Pydantic dict — no more string parsing.

In [13]:
# ── Creative Brief ─────────────────────────────────────────────
print("CREATIVE BRIEF")
print(json.dumps(results["creative_brief"], indent=2))

CREATIVE BRIEF
{
  "campaign_name": "Bali\u2019s Pure Flow",
  "tagline_concept": "Refresh Sustainably, Live Consciously",
  "core_brand_message": "Embrace Bali's natural beauty with every sip from your eco-friendly water bottle, protecting the planet while enjoying pure hydration.",
  "visual_direction": "Breezy island aesthetic with earthy tones of turquoise and green, featuring lush tropical imagery and minimalist design that highlights the product\u2019s eco-friendliness.",
  "objectives": [
    "Raise awareness of the eco-friendly water bottle among Bali's tourists and locals",
    "Position the brand as a leader in sustainable living",
    "Drive sales through engaging social media campaigns and local partnerships"
  ]
}


In [14]:
# ── Campaign Strategy ──────────────────────────────────────────
print("CAMPAIGN STRATEGY")
print(json.dumps(results["strategy"], indent=2))

CAMPAIGN STRATEGY
{
  "target_audience": "The target audience includes environmentally conscious tourists aged 25-45 visiting Bali, as well as local residents who value sustainability. They are active lifestyle enthusiasts interested in wellness, outdoor activities, and eco-friendly products. This group typically seeks authentic experiences that align with their values and preferences for sustainable living.",
  "messaging_pillars": [
    "Sustainability: Highlight the product's eco-friendly materials and impact on reducing plastic waste.",
    "Lifestyle: Position the water bottle as an essential accessory for an active, health-conscious lifestyle in Bali's natural surroundings.",
    "Community: Foster a connection with both tourists and locals by promoting a shared commitment to protecting Bali's unique environment."
  ],
  "recommended_channels": [
    "Instagram: Leveraging visual storytelling and user-generated content to engage the audience.",
    "Local Partnerships: Collaborat

In [15]:
# ── Ad Copy ────────────────────────────────────────────────────
print("AD COPY")
print(json.dumps(results["copy"], indent=2))

AD COPY
{
  "hero_tagline": "Refresh Sustainably, Live Consciously",
  "instagram_caption": "Sip mindfully in Bali's paradise \ud83c\udf0a\ud83c\udf3f #EcoFriendly #SustainableLiving #PureHydration #BaliLife #JoinTheMovement",
  "video_script": "[Scene of pristine Bali waters] \"Quench your thirst while preserving paradise\" [Cut to product shot] \"Grab your eco-friendly bottle today!\"",
  "email_subject": "Your Sustainable Adventure Starts Here!",
  "email_preview": "Discover the eco-friendly water bottle that makes a difference in Bali.",
  "print_headline": "Embrace Bali with Every Sip!",
  "print_body": "Join the movement for a sustainable future. Our eco-friendly water bottles keep you hydrated while protecting the beauty of Bali. Each sip supports local artisans and reduces plastic waste. Make a change\u2014choose wisely!"
}


In [16]:
# ── Media Plan ─────────────────────────────────────────────────
print("MEDIA PLAN")
print(json.dumps(results["media_plan"], indent=2))

MEDIA PLAN
{
  "total_budget_usd": 50000,
  "channel_allocations": [
    {
      "channel": "Instagram",
      "percent_of_budget": 40
    },
    {
      "channel": "Local Partnerships",
      "percent_of_budget": 30
    },
    {
      "channel": "Facebook",
      "percent_of_budget": 20
    },
    {
      "channel": "Influencer Networks",
      "percent_of_budget": 10
    }
  ],
  "campaign_timeline_weeks": 8,
  "kpis": [
    "Reach 100,000 people on social media",
    "Achieve a 5% engagement rate on Instagram posts",
    "Generate 500 pre-orders through local partnerships",
    "Increase email open rate to 25%"
  ],
  "launch_milestones": [
    "Week 1: Finalize creative assets and partnerships",
    "Week 2: Launch Instagram and Facebook ads",
    "Week 3: Host an influencer event in Bali",
    "Week 4: Roll out local partnerships and refill stations",
    "Week 5-6: Monitor and optimize ad performance",
    "Week 7: Start email marketing campaign",
    "Week 8: Evaluate campaign p

## 11. Guardrail Validation Demo

Confirm that bad prompts are rejected *before* any API call is made.

In [17]:
print("Testing guardrails...\n")

bad_prompts = [
    ("",                                     "empty prompt"),
    ("hi",                                   "too short"),
    ("x" * 501,                              "too long"),
    ("Launch a scam product in the market",  "banned keyword"),
]

for prompt, label in bad_prompts:
    try:
        validate_campaign_prompt(prompt)
        print(f"  [{label}] ERROR: should have been rejected")
    except ValueError as e:
        print(f"  [{label}] OK — blocked: {e}")

print("\nAll guardrail tests passed.")

Testing guardrails...

  [empty prompt] OK — blocked: Campaign prompt too short (0 chars). Provide at least 10 characters.
  [too short] OK — blocked: Campaign prompt too short (2 chars). Provide at least 10 characters.
  [too long] OK — blocked: Campaign prompt too long (501 chars). Keep it under 500 characters.
  [banned keyword] OK — blocked: Campaign prompt contains disallowed content: ['scam']

All guardrail tests passed.


## 12. Save Structured Output to JSON

In [18]:
output_path = "sample_output_v2.json"

output_payload = {
    "campaign_prompt": TEST_PROMPT,
    "pipeline_version": "v2",
    "agents": [
        "Creative Director",
        "Strategist",
        "Copywriter",
        "Media Planner",
    ],
    "creative_brief":  results.get("creative_brief"),
    "strategy":        results.get("strategy"),
    "copy":            results.get("copy"),
    "media_plan":      results.get("media_plan"),
    "token_usage":     results.get("token_usage"),
}

with open(output_path, "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"Structured output saved to: {output_path}")
print(f"Top-level keys: {list(output_payload.keys())}")

Structured output saved to: sample_output_v2.json
Top-level keys: ['campaign_prompt', 'pipeline_version', 'agents', 'creative_brief', 'strategy', 'copy', 'media_plan', 'token_usage']


## Summary: What Changed from v1

| Feature | v1 | v2 |
|---|---|---|
| Output format | Raw text strings | Pydantic models (`output_type`) |
| Agent instructions | Bullet-list prompts | Role / Goal / Backstory pattern |
| Agent count | 3 | **4** (+ Media Planner) |
| `handoff` usage | Imported, unused | Wired on every agent |
| Error handling | None | Retry with exponential backoff |
| Input validation | None | Guardrail (length + content) |
| Token tracking | None | Per-agent + total cost estimate |
| Output file | Plain `.txt` | Structured `.json` |

### How to Extend Further
- Add a **Brand Guardian** agent to validate all copy against brand guidelines
- Attach **tools** (web search, social trend APIs) via `tools=[...]` on any agent
- Enable **memory** between runs by persisting `results` and injecting prior campaigns as context
- Switch to **hierarchical orchestration** by promoting the Creative Director to an orchestrator that spawns and delegates to the other agents